# 64 — Test Agent 62 vs Random

Runs `62-One_angle_polars.py` (agent 62) against a random opponent for N_STEPS steps, collecting:
- `df_raws` — list of pandas DataFrames from `_simulate` (one per step)
- `df_s` — list of Polars DataFrames (`attacks_with_angle`) from `take_action(..., return_df=True)`

In [7]:
import importlib.util, sys, random, math
import kaggle_environments as ke
import polars as pl
import pandas as pd

spec = importlib.util.spec_from_file_location("agent62", "62-One_angle_polars.py")
m = importlib.util.module_from_spec(spec)
spec.loader.exec_module(m)
m.step = 0
m.num_agents = None
m.player_id = None

NB_STEPS_SIM = m.NB_STEPS_SIM
print(f"Loaded agent62, NB_STEPS_SIM={NB_STEPS_SIM}")

Loaded agent62, NB_STEPS_SIM=10


In [8]:
SEED = 42
N_STEPS = 100
random.seed(SEED)

def random_agent_fn(obs):
    player = obs.player
    my_planets = [p for p in obs.planets if p[1] == player]
    if not my_planets:
        return []
    planet = random.choice(my_planets)
    ships = planet[5] // 2
    if ships < 1:
        return []
    return [[planet[0], random.uniform(0, 2 * math.pi), ships]]

In [9]:
df_raws = []
df_s = []
snaps = []

env = ke.make("orbit_wars", debug=False)
env.reset(2)

for env_step in range(N_STEPS):
    obs0 = env.state[0].observation
    obs1 = env.state[1].observation

    snaps.append({
        'step':    env_step,
        'planets': [list(p) for p in obs0.planets],
        'fleets':  [list(f) for f in obs0.fleets],
    })

    df = m._simulate(obs0, global_step=env_step, num_agents=2, n_steps=NB_STEPS_SIM)
    df_raws.append(df)

    action0, df_ = m.take_action(df, player_id=0, nb_steps_sim=NB_STEPS_SIM, return_df=True)
    df_s.append(df_)

    action1 = random_agent_fn(obs1)
    env.step([action0, action1])
    if env.state[0].status != "ACTIVE":
        break

obs0 = env.state[0].observation
snaps.append({
    'step':    env_step + 1,
    'planets': [list(p) for p in obs0.planets],
    'fleets':  [list(f) for f in obs0.fleets],
})

p0_ships = sum(p[5] for p in obs0.planets if p[1] == 0)
p1_ships = sum(p[5] for p in obs0.planets if p[1] == 1)
print(f"Player 0 (agent 62): {p0_ships} ships on planets")
print(f"Player 1 (random):   {p1_ships} ships on planets")
winner = "Agent 62 wins" if p0_ships > p1_ships else "Random wins" if p1_ships > p0_ships else "Tie"
print(f"After {env_step + 1} steps: {winner}")
print(f"Collected {len(df_raws)} df_raws, {len(df_s)} df_s, {len(snaps)} snaps")

From 4, To 20 at step 10 with 13 ships (target has min 13)
From 4, To 14 at step 23 with 9 ships (target has min 10)
From 20, To 16 at step 19 with 19 ships (target has min 21)
From 20, To 0 at step 22 with 16 ships (target has min 17)
From 20, To 12 at step 28 with 12 ships (target has min 16)
From 16, To 28 at step 27 with 14 ships (target has min 17)
From 4, To 22 at step 32 with 13 ships (target has min 13)
From 20, To 26 at step 33 with 25 ships (target has min 29)
From 20, To 5 at step 38 with 11 ships (target has min 14)
From 20, To 17 at step 43 with 21 ships (target has min 28)
From 14, To 27 at step 46 with 25 ships (target has min 25)
From 12, To 17 at step 43 with 19 ships (target has min 20)
From 22, To 2 at step 47 with 16 ships (target has min 21)
From 17, To 29 at step 47 with 14 ships (target has min 22)
From 16, To 8 at step 47 with 84 ships (target has min 87)
From 17, To 1 at step 54 with 17 ships (target has min 20)
From 20, To 9 at step 54 with 84 ships (target ha

In [10]:
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML

_COLORS = {0: 'steelblue', 1: 'tomato', -1: '#888888'}

def make_animation(snapshots, title='', interval=150):
    fig, ax = plt.subplots(figsize=(6, 6))
    fig.patch.set_facecolor('#111122')

    def draw(frame):
        snap = snapshots[frame]
        ax.cla()
        ax.set_xlim(0, 100)
        ax.set_ylim(100, 0)
        ax.set_aspect('equal')
        ax.set_facecolor('#111122')
        ax.tick_params(colors='#aaaaaa')
        for sp in ax.spines.values():
            sp.set_edgecolor('#444444')
        ax.set_title(f"{title}  (step {snap['step']})", color='white', fontsize=11)

        ax.add_patch(plt.Circle((50, 50), 10, color='gold', zorder=2, alpha=0.9))

        for p in snap['planets']:
            pid, owner, x, y, radius, ships, production = p
            c = _COLORS.get(owner, '#888888')
            ax.add_patch(plt.Circle((x, y), radius, color=c, alpha=0.85, zorder=3))
            ax.text(x, y, str(ships), ha='center', va='center',
                    color='white', fontsize=7, fontweight='bold', zorder=4)
            ax.text(x, y + 2, str(pid), ha='center', va='center',
                    color='red', fontsize=7, fontweight='bold', zorder=4)
            ax.text(x, y - 2, '+' + str(production), ha='center', va='center',
                    color='green', fontsize=7, fontweight='bold', zorder=4)

        for f in snap['fleets']:
            fid, owner, x, y, angle, from_id, ships = f
            c = _COLORS.get(owner, '#888888')
            ax.plot(x, y, 'D', color=c, markersize=5, zorder=5)
            ax.text(x + 1.5, y + 1.5, str(ships), color=c, fontsize=5, zorder=6)

        return []

    ani = animation.FuncAnimation(fig, draw, frames=len(snapshots), interval=interval)
    plt.close()
    return HTML(ani.to_jshtml())

In [11]:
make_animation(snaps, title='Agent 62 vs Random', interval=100)

## Inspect a specific step

Change `STEP` to any step index you want to examine.

In [12]:
STEP = 0

print(f"=== df_raws[{STEP}] — pandas, _simulate output ===")
print(f"Shape: {df_raws[STEP].shape}")
df_raws[STEP]

=== df_raws[0] — pandas, _simulate output ===
Shape: (352, 9)


,step,id,x,y,radius,ships,production,owner,nature
0,0,0,71.701691,95.809853,1.000000,15,1,-1,fix
1,0,1,4.190147,71.701691,1.000000,15,1,-1,fix
2,0,2,95.809853,28.298309,1.000000,15,1,-1,fix
3,0,3,28.298309,4.190147,1.000000,15,1,-1,fix
4,0,4,96.107892,67.113463,1.000000,10,1,0,fix
...,...,...,...,...,...,...,...,...,...
347,10,27,43.614205,15.780973,1.693147,24,2,-1,moving
348,10,28,92.510567,93.303792,1.000000,12,1,-1,fix
349,10,29,6.696208,92.510567,1.000000,12,1,-1,fix
350,10,30,93.303792,7.489433,1.000000,12,1,-1,fix


In [13]:
STEP = 0

print(f"=== df_s[{STEP}] — Polars, attacks_with_angle from take_action ===")
print(f"Shape: {df_s[STEP].shape}")
df_s[STEP]

=== df_s[0] — Polars, attacks_with_angle from take_action ===
Shape: (158, 32)


id_src,step_src,x_src,y_src,radius_src,ships_min,production_src,nature_src,owner_src,row_count,is_mine,ships_sent,step,id,x,y,radius,ships,production,owner,nature,dist_tgt_src,step_diff,fleet_speed,dist_fleet_src_min,dist_fleet_src_max,collision,angle,radius_angle,angle_min,angle_max,final_angle
i64,i64,f64,f64,f64,i64,i64,str,i64,u32,i32,i64,i64,i64,f64,f64,f64,i64,i64,i64,str,f64,i64,f64,f64,f64,bool,f64,f64,f64,f64,f64
4,0,96.107892,67.113463,1.0,10,1,"""fix""",0,11,11,3,9,16,88.55429,81.895369,2.386294,18,4,-1,"""fix""",16.600049,9,1.317126,12.95413,14.271256,true,2.043208,0.033831,2.009377,2.077039,2.043208
4,0,96.107892,67.113463,1.0,10,1,"""fix""",0,11,11,4,9,16,88.55429,81.895369,2.386294,18,4,-1,"""fix""",16.600049,9,1.449519,14.145668,15.595187,true,2.043208,0.134622,1.908585,2.17783,2.043208
4,0,96.107892,67.113463,1.0,10,1,"""fix""",0,11,11,3,10,16,88.55429,81.895369,2.386294,18,4,-1,"""fix""",16.600049,10,1.317126,14.271256,15.588381,true,2.043208,0.134454,1.908754,2.177662,2.043208
4,0,96.107892,67.113463,1.0,10,1,"""fix""",0,11,11,4,10,16,88.55429,81.895369,2.386294,18,4,-1,"""fix""",16.600049,10,1.449519,15.595187,17.044706,true,2.043208,0.139493,1.903715,2.182701,2.043208
4,0,96.107892,67.113463,1.0,10,1,"""fix""",0,11,11,6,7,16,88.55429,81.895369,2.386294,18,4,-1,"""fix""",16.600049,7,1.660517,12.723621,14.384138,true,2.043208,0.057314,1.985894,2.100521,2.043208
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
4,0,96.107892,67.113463,1.0,10,1,"""fix""",0,11,11,18,10,26,84.219027,43.614205,1.693147,24,2,-1,"""moving""",26.335531,10,2.353303,24.633033,26.986337,true,-2.039173,0.058641,4.185372,4.302653,-2.039173
4,0,96.107892,67.113463,1.0,10,1,"""fix""",0,11,11,18,10,28,92.510567,93.303792,1.0,12,1,-1,"""fix""",26.436227,10,2.353303,24.633033,26.986337,true,1.707295,0.031267,1.676029,1.738562,1.707295
4,0,96.107892,67.113463,1.0,10,1,"""fix""",0,11,11,19,10,26,84.219027,43.614205,1.693147,24,2,-1,"""moving""",26.335531,10,2.391453,25.014526,27.405979,true,-2.039173,0.048834,4.195178,4.292847,-2.039173


## Summary: non-empty steps and attack counts

In [14]:
non_empty = [(i, df_s[i].shape[0]) for i in range(len(df_s)) if not df_s[i].is_empty()]
print(f"Steps with attacks: {len(non_empty)} / {len(df_s)}")
if non_empty:
    counts = [n for _, n in non_empty]
    print(f"Rows per step — min: {min(counts)}, max: {max(counts)}, mean: {sum(counts)/len(counts):.1f}")

Steps with attacks: 100 / 100
Rows per step — min: 36, max: 36799, mean: 12023.2
